In [234]:
import pandas as pd
import torch
import numpy as np
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
from pathlib import Path
import mlflow
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [235]:
df = pd.read_excel(
    "../data/Concrete_Data.xls",
    engine="xlrd"
)

display(df.head())
display(df.head())
display(df.info())
display(df.describe())
print("Missing values:\n", df.isnull().sum())

,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


<class 'pandas.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 9 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   Cement (component 1)(kg in a m^3 mixture)              1030 non-null   float64
 1   Blast Furnace Slag (component 2)(kg in a m^3 mixture)  1030 non-null   float64
 2   Fly Ash (component 3)(kg in a m^3 mixture)             1030 non-null   float64
 3   Water  (component 4)(kg in a m^3 mixture)              1030 non-null   float64
 4   Superplasticizer (component 5)(kg in a m^3 mixture)    1030 non-null   float64
 5   Coarse Aggregate  (component 6)(kg in a m^3 mixture)   1030 non-null   float64
 6   Fine Aggregate (component 7)(kg in a m^3 mixture)      1030 non-null   float64
 7   Age (day)                                              1030 non-null   int64  
 8   Concrete compressive strength(MPa, megapascals)        1030 

None

,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.165631,73.895485,54.187136,181.566359,6.203112,972.918592,773.578883,45.662136,35.817836
std,104.507142,86.279104,63.996469,21.355567,5.973492,77.753818,80.175427,63.169912,16.705679
min,102.000000,0.000000,0.000000,121.750000,0.000000,801.000000,594.000000,1.000000,2.331808
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.707115
50%,272.900000,22.000000,0.000000,185.000000,6.350000,968.000000,779.510000,28.000000,34.442774
75%,350.000000,142.950000,118.270000,192.000000,10.160000,1029.400000,824.000000,56.000000,46.136287
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.599225


Missing values:
 Cement (component 1)(kg in a m^3 mixture)                0
Blast Furnace Slag (component 2)(kg in a m^3 mixture)    0
Fly Ash (component 3)(kg in a m^3 mixture)               0
Water  (component 4)(kg in a m^3 mixture)                0
Superplasticizer (component 5)(kg in a m^3 mixture)      0
Coarse Aggregate  (component 6)(kg in a m^3 mixture)     0
Fine Aggregate (component 7)(kg in a m^3 mixture)        0
Age (day)                                                0
Concrete compressive strength(MPa, megapascals)          0
dtype: int64


In [236]:
df.columns = df.columns.str.strip() # this fixes the trailing spaces for the column names
x = df.drop(columns=['Concrete compressive strength(MPa, megapascals)'])
y= df['Concrete compressive strength(MPa, megapascals)']

print("Features shape:", x.shape)
print("Target shape:", y.shape)

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

Features shape: (1030, 8)
Target shape: (1030,)


In [237]:


# Convert features to PyTorch tensors

if not isinstance(x_train, torch.Tensor):
    x_train = torch.tensor(x_train, dtype=torch.float32)

if not isinstance(x_test, torch.Tensor):
    x_test = torch.tensor(x_test, dtype=torch.float32)

if not isinstance(y_train, torch.Tensor):
    y_train = torch.tensor(y_train.to_numpy(), dtype = torch.float32).reshape(-1,1)

if not isinstance(y_test, torch.Tensor):
    y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)


x_train: torch.Size([824, 8])
y_train: torch.Size([824, 1])
x_test: torch.Size([206, 8])
y_test: torch.Size([206, 1])


In [238]:
torch.manual_seed(42)
class ConcreteStrenthModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.network= nn.Sequential(
            nn.Linear(8, 25),
            nn.ReLU(),
            nn.Linear(25, 10),
            nn.ReLU(),
            nn.Linear(10,1)
        )

    def forward(self, x):
        return self.network(x)

model = ConcreteStrenthModel()
print(model)

model.eval()
with torch.no_grad():
    predictions = model(x_train[:5])

print(f"Predictions: {predictions}")
print(f"Actual: {y_train[:5]}")

ConcreteStrenthModel(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=25, bias=True)
    (1): ReLU()
    (2): Linear(in_features=25, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=1, bias=True)
  )
)
Predictions: tensor([[0.0823],
        [0.1342],
        [0.1110],
        [0.0995],
        [0.1522]])
Actual: tensor([[27.6811],
        [62.0528],
        [23.8007],
        [33.3982],
        [ 7.3981]])


In [239]:
loss_fn = nn.L1Loss()
with torch.no_grad():
    initial_predictions = model(x_train)
    initial_loss = loss_fn(initial_predictions, y_train)

print(f"Initial Loss: {initial_loss.item()}")

Initial Loss: 35.79055404663086


In [240]:
#declare the optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr = 0.001
)


In [241]:

# Create the database in your project folder
project_dir = Path.cwd().parent
database_path = project_dir / "mlflow.db"

# Tell MLflow to use SQLite
mlflow.set_tracking_uri(f"sqlite:///{database_path.as_posix()}")

# Create or select the experiment
mlflow.set_experiment("Concrete Strength - PyTorch")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment ready!")

Tracking URI: sqlite:///c:/Users/Mumba Mumba/Desktop/tensor flow/myfirstlesson_on_nn/mlflow.db
Experiment ready!


In [242]:
epochs = 300

with mlflow.start_run(run_name='MAE Loss - SGD 0.001'):
    mlflow.log_param("loss_function", 'L1Loss')
    mlflow.log_param("optimizer", "SGD")
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("epocchs", epochs)
    mlflow.log_param("architecture", "8-25-10-1")

#declaring the training loop
for epoch in range(epochs):
    model.train()
    predictions = model(x_train)

    loss = loss_fn(predictions, y_train)

    optimizer.zero_grad(0)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")


model.eval()
with torch.no_grad():
    test_predictions = model(x_test).cpu().numpy().flatten()

actual = y_test.cpu().numpy().flatten()

test_mae = mean_absolute_error(actual, test_predictions)
test_rmse = np.sqrt(mean_squared_error(actual, test_predictions))
test_r2 = r2_score(actual, test_predictions)


# Save the final evaluation metrics
mlflow.log_metric("test_mae", test_mae)
mlflow.log_metric("test_rmse", test_rmse)
mlflow.log_metric("test_r2", test_r2)

print(f"Test MAE:  {test_mae:.2f} MPa")
print(f"Test RMSE: {test_rmse:.2f} MPa")
print(f"Test R²:   {test_r2:.3f}")

Epoch 10/300, Loss: 35.7790
Epoch 20/300, Loss: 35.7661
Epoch 30/300, Loss: 35.7532
Epoch 40/300, Loss: 35.7403
Epoch 50/300, Loss: 35.7273
Epoch 60/300, Loss: 35.7142
Epoch 70/300, Loss: 35.7011
Epoch 80/300, Loss: 35.6880
Epoch 90/300, Loss: 35.6748
Epoch 100/300, Loss: 35.6616
Epoch 110/300, Loss: 35.6482
Epoch 120/300, Loss: 35.6348
Epoch 130/300, Loss: 35.6214
Epoch 140/300, Loss: 35.6078
Epoch 150/300, Loss: 35.5942
Epoch 160/300, Loss: 35.5805
Epoch 170/300, Loss: 35.5667
Epoch 180/300, Loss: 35.5528
Epoch 190/300, Loss: 35.5388
Epoch 200/300, Loss: 35.5248
Epoch 210/300, Loss: 35.5107
Epoch 220/300, Loss: 35.4964
Epoch 230/300, Loss: 35.4821
Epoch 240/300, Loss: 35.4677
Epoch 250/300, Loss: 35.4531
Epoch 260/300, Loss: 35.4385
Epoch 270/300, Loss: 35.4237
Epoch 280/300, Loss: 35.4087
Epoch 290/300, Loss: 35.3936
Epoch 300/300, Loss: 35.3783
Test MAE:  35.17 MPa
Test RMSE: 38.66 MPa
Test R²:   -4.801


In [243]:
for name, parameter in model.named_parameters():
    print(name)
    print(parameter.detach())

network.0.weight
tensor([[ 0.2700,  0.2930, -0.0818,  0.3239, -0.0764,  0.0711, -0.1713,  0.2062],
        [ 0.3127, -0.2577,  0.3050,  0.0676,  0.2611,  0.0460,  0.1700, -0.0496],
        [ 0.2726,  0.0538, -0.1649,  0.0900, -0.1615, -0.0440, -0.1431,  0.2354],
        [-0.2802, -0.1624, -0.0991, -0.2139,  0.0343, -0.3494,  0.3207, -0.3004],
        [ 0.2694,  0.0605, -0.1124,  0.2184,  0.0560,  0.2854,  0.0390, -0.1092],
        [ 0.0952, -0.0978,  0.1518,  0.3160,  0.2077, -0.1599,  0.2074,  0.0635],
        [ 0.1808, -0.2163, -0.3505, -0.1368, -0.2716,  0.2910,  0.1011,  0.1457],
        [ 0.1121, -0.0063,  0.2773, -0.2516,  0.0234, -0.2425,  0.1095, -0.1220],
        [ 0.1085, -0.0724,  0.2917, -0.2081, -0.2116, -0.2114,  0.3169,  0.1173],
        [ 0.3440, -0.2938, -0.3541, -0.2769, -0.2393,  0.1444,  0.1268,  0.2935],
        [-0.1866, -0.2443,  0.1964, -0.1478,  0.2204, -0.0876,  0.2084, -0.2778],
        [-0.1800,  0.1083,  0.0767, -0.0901,  0.2119,  0.2395, -0.2563, -0.1886],

In [244]:
#model evaluations
model.eval()
with torch.no_grad():
    test_predictions = model(x_test)

print("First 10 predictions:")
print(test_predictions[:10])

print("\nFirst 10 actual values:")
print(y_test[:10])

with torch.no_grad():
    test_loss = loss_fn(test_predictions, y_test)

print('Test MSE:', test_loss.item())

First 10 predictions:
tensor([[0.3969],
        [0.3402],
        [0.3717],
        [0.3412],
        [0.8320],
        [0.4453],
        [0.4611],
        [0.6136],
        [0.7005],
        [0.4560]])

First 10 actual values:
tensor([[52.9083],
        [55.8958],
        [74.4979],
        [35.3012],
        [10.5352],
        [44.2844],
        [23.6904],
        [45.3675],
        [37.4041],
        [48.8466]])
Test MSE: 35.171939849853516


In [245]:
comparison = pd.DataFrame({
    "Actual": y_test[:20].flatten().numpy(),
    "Predicted": test_predictions[:20].flatten().numpy()
})

comparison["Error"] = comparison["Predicted"] - comparison["Actual"]

display(comparison)

,Actual,Predicted,Error
0,52.908321,0.396890,-52.511433
1,55.895821,0.340217,-55.555603
2,74.497879,0.371718,-74.126160
3,35.301170,0.341171,-34.959999
4,10.535193,0.831957,-9.703236
5,44.284355,0.445345,-43.839008
6,23.690395,0.461095,-23.229301
7,45.367519,0.613644,-44.753876
8,37.404072,0.700536,-36.703537
9,48.846615,0.455979,-48.390636


In [250]:
import mlflow

print(mlflow.get_tracking_uri())

sqlite:///c:/Users/Mumba Mumba/Desktop/tensor flow/myfirstlesson_on_nn/mlflow.db
